# CS 229 - Homework 0: Generated Name Detector

Submit **PDF** of completed IPython notebook on Canvas.

**Maximum points**: 10

<div style="margin-bottom: 15px; padding: 15px; color: #31708f; background-color: #d9edf7; border: 1px solid #bce8f1; border-radius: 5px;">
    
<b><font size=+2>Enter your information below:</font></b><br/><br/>

  <b>(full) Name</b>: Vedant Deepak Borkute
  <br/>
  <b>Student ID Number</b>:  862552981
  <br/><br/>
    
<b>By submitting this notebook, I assert that the work below is my own work, completed for this course.  Except where explicitly cited, none of the portions of this notebook are duplicated from anyone else's work.</b>
</div>

# Overview

In this assignment you will build a classifier that distinguishes **curated names** from **generated names**. This will test your ability to:

- Tokenize text data for use with neural networks
- Define a Multi-Layer Perceptron (MLP) in PyTorch
- Train a model using SGD and Cross-Entropy loss
- Evaluate your model on held-out validation data

### Background

Our dataset is inspired by Andrej Karpathy's [makemore](https://github.com/karpathy/makemore) project. One set of names comes from the Social Security Administration's curated list of baby names. The other set was produced by a character-level bigram model trained on that same list — these names are *generated* by a simple statistical model and often have telltale artifacts.

**Your job**: given a name (a string of lowercase characters), predict whether it came from the **curated list** (label `1`) or was **model-generated** (label `0`).

All names use only lowercase letters `a-z` and range from 2 to 15 characters. We will pad all names to a fixed maximum length of 15.

Complete all parts marked `TODO` and ensure all test cells produce the expected output.

In [15]:
import torch
import torch.nn as nn
from torch.optim import SGD
from torch.utils.data import DataLoader, TensorDataset

## Load Data
Download `hw0_data.pt` from Canvas and place it in the same directory as this notebook. The file contains 20,000 names split into training (16,000) and validation (4,000) sets, with binary labels: `1` = curated, `0` = generated.

In [16]:
data = torch.load('hw0_data.pt', weights_only=False)
train_names = data['train_names']      # list of 16000 strings
train_labels = data['train_labels']    # tensor of shape (16000,)
val_names = data['val_names']          # list of 4000 strings
val_labels = data['val_labels']        # tensor of shape (4000,)

print(f"Train: {len(train_names)} names ({train_labels.sum().item()} curated, {(train_labels == 0).sum().item()} generated)")
print(f"Val:   {len(val_names)} names ({val_labels.sum().item()} curated, {(val_labels == 0).sum().item()} generated)")
print(f"\nSample curated names:  {[n for n, l in zip(train_names[:50], train_labels[:50]) if l == 1][:5]}")
print(f"Sample generated names: {[n for n, l in zip(train_names[:50], train_labels[:50]) if l == 0][:5]}")

Train: 16000 names (7995 curated, 8005 generated)
Val:   4000 names (2005 curated, 1995 generated)

Sample curated names:  ['pieper', 'lucie', 'eriksen', 'jad', 'anara']
Sample generated names: ['jieron', 'agh', 'slan', 'keyanvexy', 'kiy']


## Part 1: Character Tokenizer [2 points]

Define a `tokenize` function that converts a name string into a padded integer tensor.

**Requirements:**
- Map each lowercase letter to an integer: `'a'` → 1, `'b'` → 2, ..., `'z'` → 26
- Use `0` as the **padding token** (to pad shorter names to `MAX_LEN`)
- The output should be a 1-D `torch.long` tensor of length `MAX_LEN`
- Characters are placed left-aligned, padding fills the right

For example, `tokenize('cat')` with `MAX_LEN=5` should return: `tensor([3, 1, 20, 0, 0])`

In [17]:
MAX_LEN = 15  # Maximum name length in our dataset

def tokenize(name):
    """Convert a lowercase name string to a padded integer tensor.

    Args:
        name: string of lowercase letters (length <= MAX_LEN)
    Returns:
        torch.long tensor of shape (MAX_LEN,)
    """
    # TODO [2 points]: implement the tokenizer
    ordMap = []
    for i in name:
      ordMap.append(ord(i) - 96)
    if len(ordMap) < MAX_LEN:
      for i in range(MAX_LEN - len(ordMap)):
        ordMap.append(0)
    return torch.tensor(ordMap)


### Test your tokenizer
The cell below should print:
```
tensor([ 1,  2,  3, 24, 25, 26,  0,  0,  0,  0,  0,  0,  0,  0,  0])
torch.Size([15]) torch.int64
tensor([ 3,  1, 20,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0])
```

In [18]:
out = tokenize('abcxyz')
print(out)
print(out.shape, out.dtype)
print(tokenize('cat'))

tensor([ 1,  2,  3, 24, 25, 26,  0,  0,  0,  0,  0,  0,  0,  0,  0])
torch.Size([15]) torch.int64
tensor([ 3,  1, 20,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0])


## Part 2: Understanding Embeddings [1 point]

Before building our MLP, let's get hands-on with `nn.Embedding`, which maps integer token IDs to dense vectors.

Create an `nn.Embedding` layer with vocabulary size 27 (26 letters + 1 pad token) and embedding dimension **2**. Use your tokenizer to tokenize the name `'hello'` and pass the result through the embedding layer. Print the output — since the embedding dimension is 2, each token maps to a 2-D point, which you could imagine plotting on a plane.

**Note**: We use vocab size 27 (not 26) because our pad token at index 0 also needs an embedding vector.

<svg xmlns="http://www.w3.org/2000/svg" width="720" height="260" font-family="monospace, sans-serif">
  <rect width="720" height="260" fill="#fafafa" rx="10"/>
  <text x="360" y="26" text-anchor="middle" font-size="15" font-weight="bold" fill="#333">nn.Embedding — turning token IDs into dense vectors</text>

  <!-- Column 1: characters -->
  <text x="68" y="56" text-anchor="middle" font-size="12" fill="#666">token IDs</text>
  <rect x="20"  y="64" width="96" height="28" rx="5" fill="#e8f4fd" stroke="#90caf9" stroke-width="1.5"/>
  <rect x="20"  y="98" width="96" height="28" rx="5" fill="#e8f4fd" stroke="#90caf9" stroke-width="1.5"/>
  <rect x="20" y="132" width="96" height="28" rx="5" fill="#e8f4fd" stroke="#90caf9" stroke-width="1.5"/>
  <rect x="20" y="166" width="96" height="28" rx="5" fill="#e8f4fd" stroke="#90caf9" stroke-width="1.5"/>
  <rect x="20" y="200" width="96" height="28" rx="5" fill="#ecf5fd" stroke="#b3d9f6" stroke-width="1.5" stroke-dasharray="5,3"/>
  <text x="68"  y="83" text-anchor="middle" font-size="13" fill="#1565c0">'h' → 8</text>
  <text x="68" y="117" text-anchor="middle" font-size="13" fill="#1565c0">'e' → 5</text>
  <text x="68" y="151" text-anchor="middle" font-size="13" fill="#1565c0">'l' → 12</text>
  <text x="68" y="185" text-anchor="middle" font-size="13" fill="#1565c0">'l' → 12</text>
  <text x="68" y="219" text-anchor="middle" font-size="12" fill="#aaa">pad → 0</text>

  <!-- Embedding table -->
  <text x="292" y="56" text-anchor="middle" font-size="12" fill="#666">Embedding table  (27 rows × embed_dim cols)</text>
  <rect x="182" y="64" width="220" height="172" rx="6" fill="#fff8e1" stroke="#ffca28" stroke-width="1.5"/>
  <rect x="188" y="70"  width="208" height="22" rx="3" fill="#fff3cd"/>
  <rect x="188" y="96"  width="208" height="22" rx="3" fill="#fffde7"/>
  <rect x="188" y="120" width="208" height="22" rx="3" fill="#fff3cd"/>
  <rect x="188" y="144" width="208" height="22" rx="3" fill="#fffde7"/>
  <rect x="188" y="168" width="208" height="22" rx="3" fill="#fff3cd" stroke="#ffa000" stroke-width="1.5"/>
  <rect x="188" y="192" width="208" height="22" rx="3" fill="#fffde7"/>
  <rect x="188" y="218" width="208" height="12" rx="3" fill="#fafafa"/>
  <text x="194"  y="85" font-size="11" fill="#999">row 0  (pad):      [ 0.00,  0.00, … ]</text>
  <text x="194" y="111" font-size="11" fill="#999">rows 1–4  (a–d):   [ … ]</text>
  <text x="194" y="135" font-size="11" fill="#666" font-weight="bold">row 5  (e):        [ 1.59,  0.29, … ]</text>
  <text x="194" y="159" font-size="11" fill="#999">rows 6–7  (f–g):   [ … ]</text>
  <text x="194" y="183" font-size="11" fill="#0d47a1" font-weight="bold">row 8  (h):        [ 0.68,  0.54, … ]</text>
  <text x="194" y="207" font-size="11" fill="#999">rows 9–26  …:      [ … ]</text>
  <text x="292" y="231" text-anchor="middle" font-size="11" fill="#bbb">⋮</text>

  <!-- Arrows from tokens to table rows -->
  <defs>
    <marker id="blarr"  markerWidth="8" markerHeight="8" refX="7" refY="3" orient="auto"><path d="M0,0 L0,6 L8,3 z" fill="#0d47a1"/></marker>
    <marker id="greyarr" markerWidth="8" markerHeight="8" refX="7" refY="3" orient="auto"><path d="M0,0 L0,6 L8,3 z" fill="#999"/></marker>
    <marker id="grnarr" markerWidth="8" markerHeight="8" refX="7" refY="3" orient="auto"><path d="M0,0 L0,6 L8,3 z" fill="#388e3c"/></marker>
  </defs>
  <line x1="118" y1="79"  x2="180" y2="179" stroke="#0d47a1" stroke-width="2"   marker-end="url(#blarr)"  stroke-dasharray="5,3"/>
  <line x1="118" y1="113" x2="180" y2="131" stroke="#777"    stroke-width="1.5" marker-end="url(#greyarr)" stroke-dasharray="4,3"/>

  <!-- Output vectors -->
  <text x="620" y="56" text-anchor="middle" font-size="12" fill="#666">output  (15 × embed_dim)</text>
  <rect x="534"  y="64" width="172" height="28" rx="5" fill="#e8f5e9" stroke="#4caf50" stroke-width="2"/>
  <rect x="534"  y="98" width="172" height="28" rx="5" fill="#e8f5e9" stroke="#4caf50" stroke-width="1.5"/>
  <rect x="534" y="132" width="172" height="28" rx="5" fill="#e8f5e9" stroke="#4caf50" stroke-width="1.5"/>
  <rect x="534" y="166" width="172" height="28" rx="5" fill="#e8f5e9" stroke="#4caf50" stroke-width="1.5"/>
  <rect x="534" y="200" width="172" height="28" rx="5" fill="#f1f8e9" stroke="#aed581" stroke-width="1.5" stroke-dasharray="5,3"/>
  <text x="620"  y="83" text-anchor="middle" font-size="12" fill="#2e7d32" font-weight="bold">[ 0.68,  0.54, … ]</text>
  <text x="620" y="117" text-anchor="middle" font-size="12" fill="#2e7d32">[ 1.59,  0.29, … ]</text>
  <text x="620" y="151" text-anchor="middle" font-size="12" fill="#2e7d32">[ 1.15, -0.36, … ]</text>
  <text x="620" y="185" text-anchor="middle" font-size="12" fill="#2e7d32">[ 1.15, -0.36, … ]</text>
  <text x="620" y="219" text-anchor="middle" font-size="11" fill="#aaa">[ 0.00,  0.00, … ]</text>

  <!-- Arrows from table to output -->
  <line x1="404" y1="179" x2="532"  y2="79"  stroke="#0d47a1" stroke-width="2"   marker-end="url(#blarr)"  stroke-dasharray="5,3"/>
  <line x1="404" y1="131" x2="532" y2="113"  stroke="#777"    stroke-width="1.5" marker-end="url(#greyarr)" stroke-dasharray="4,3"/>

  <!-- Footer note -->
  <text x="292" y="256" text-anchor="middle" font-size="11" fill="#e65100" font-style="italic">each token ID selects one row of the table (its learned embedding vector)</text>
  <text x="620" y="256" text-anchor="middle" font-size="11" fill="#2e7d32" font-style="italic">all rows stacked → shape (15, embed_dim)</text>
</svg>

In [19]:
# TODO [1 point]: Create an Embedding(27, 2), tokenize 'hello',
# pass through the embedding, and print the result.
# You should see a tensor of shape (15, 2) — one 2-D vector per token position.
embedding = nn.Embedding(27, 2)
print(embedding(tokenize('hello')))
print(embedding(tokenize('hello')).shape)

tensor([[-2.4725,  0.2481],
        [-1.4291, -1.3751],
        [ 0.4971, -0.8155],
        [ 0.4971, -0.8155],
        [-0.2737, -0.5539],
        [-0.0742,  1.4174],
        [-0.0742,  1.4174],
        [-0.0742,  1.4174],
        [-0.0742,  1.4174],
        [-0.0742,  1.4174],
        [-0.0742,  1.4174],
        [-0.0742,  1.4174],
        [-0.0742,  1.4174],
        [-0.0742,  1.4174],
        [-0.0742,  1.4174]], grad_fn=<EmbeddingBackward0>)
torch.Size([15, 2])


## Part 3: Define the NameMLP [2 points]

Define a `NameMLP` class that takes tokenized name input and outputs class logits (curated vs. generated).

**Architecture:**
1. `nn.Embedding` layer: vocab size 27, embedding dim of your choice (we suggest 16)
2. Flatten the embedded sequence into a single vector (hint: the flattened size is `MAX_LEN * embed_dim`)
3. Hidden layer 1: `nn.Linear` → `nn.ReLU`, with 128 hidden units
4. Hidden layer 2: `nn.Linear` → `nn.ReLU`, with 64 hidden units
5. Output layer: `nn.Linear` → 2 output logits (one per class)

The `forward` method takes `input_ids` of shape `(batch_size, MAX_LEN)` with dtype `torch.long` and returns logits of shape `(batch_size, 2)`.

<svg xmlns="http://www.w3.org/2000/svg" width="760" height="290" font-family="sans-serif">
  <rect width="760" height="290" fill="#fafafa" rx="10"/>
  <text x="380" y="26" text-anchor="middle" font-size="15" font-weight="bold" fill="#333">NameMLP Architecture</text>

  <!-- Layer 1: Input -->
  <rect x="12"  y="50" width="96" height="190" rx="8" fill="#e3f2fd" stroke="#90caf9" stroke-width="2"/>
  <text x="60"  y="44" text-anchor="middle" font-size="11" fill="#666">Input</text>
  <text x="60" y="130" text-anchor="middle" font-size="13" fill="#1565c0" font-weight="bold">token IDs</text>
  <text x="60" y="170" text-anchor="middle" font-size="10" fill="#777">(batch, 15)</text>
  <text x="60" y="185" text-anchor="middle" font-size="10" fill="#777">torch.long</text>

  <!-- Layer 2: Embedding + Flatten -->
  <rect x="132" y="50" width="116" height="190" rx="8" fill="#fff8e1" stroke="#ffca28" stroke-width="2"/>
  <text x="190"  y="44" text-anchor="middle" font-size="11" fill="#666">Embed + Flatten</text>
  <text x="190" y="105" text-anchor="middle" font-size="12" fill="#e65100" font-weight="bold">Embedding</text>
  <text x="190" y="121" text-anchor="middle" font-size="10" fill="#555">(27, embed_dim)</text>
  <text x="190" y="137" text-anchor="middle" font-size="10" fill="#888">↓ (batch,15,D)</text>
  <line x1="145" y1="150" x2="235" y2="150" stroke="#ddd" stroke-width="1"/>
  <text x="190" y="167" text-anchor="middle" font-size="12" fill="#e65100" font-weight="bold">Flatten</text>
  <text x="190" y="195" text-anchor="middle" font-size="10" fill="#777">(batch, 15·D)</text>
  <text x="190" y="210" text-anchor="middle" font-size="9"  fill="#aaa">e.g. 15×16=240</text>

  <!-- Layer 3: Hidden 1 -->
  <rect x="272" y="65" width="110" height="160" rx="8" fill="#f3e5f5" stroke="#ce93d8" stroke-width="2"/>
  <text x="327"  y="59" text-anchor="middle" font-size="11" fill="#666">Hidden 1</text>
  <text x="327" y="112" text-anchor="middle" font-size="12" fill="#6a1b9a" font-weight="bold">Linear</text>
  <text x="327" y="128" text-anchor="middle" font-size="10" fill="#555">(240 → 128)</text>
  <line x1="282" y1="146" x2="372" y2="146" stroke="#ddd" stroke-width="1"/>
  <text x="327" y="163" text-anchor="middle" font-size="12" fill="#6a1b9a" font-weight="bold">ReLU</text>
  <text x="327" y="205" text-anchor="middle" font-size="10" fill="#777">(batch, 128)</text>

  <!-- Layer 4: Hidden 2 -->
  <rect x="406" y="65" width="110" height="160" rx="8" fill="#f3e5f5" stroke="#ce93d8" stroke-width="2"/>
  <text x="461"  y="59" text-anchor="middle" font-size="11" fill="#666">Hidden 2</text>
  <text x="461" y="112" text-anchor="middle" font-size="12" fill="#6a1b9a" font-weight="bold">Linear</text>
  <text x="461" y="128" text-anchor="middle" font-size="10" fill="#555">(128 → 64)</text>
  <line x1="416" y1="146" x2="506" y2="146" stroke="#ddd" stroke-width="1"/>
  <text x="461" y="163" text-anchor="middle" font-size="12" fill="#6a1b9a" font-weight="bold">ReLU</text>
  <text x="461" y="205" text-anchor="middle" font-size="10" fill="#777">(batch, 64)</text>

  <!-- Layer 5: Output linear -->
  <rect x="540" y="80" width="108" height="130" rx="8" fill="#e8f5e9" stroke="#66bb6a" stroke-width="2"/>
  <text x="594"  y="74" text-anchor="middle" font-size="11" fill="#666">Output</text>
  <text x="594" y="125" text-anchor="middle" font-size="12" fill="#2e7d32" font-weight="bold">Linear</text>
  <text x="594" y="141" text-anchor="middle" font-size="10" fill="#555">(64 → 2)</text>
  <text x="594" y="185" text-anchor="middle" font-size="10" fill="#777">(batch, 2) logits</text>

  <!-- Layer 6: two logit outputs -->
  <rect x="668" y="95" width="80" height="100" rx="8" fill="#ffebee" stroke="#ef9a9a" stroke-width="2"/>
  <text x="708" y="127" text-anchor="middle" font-size="11" fill="#b71c1c">logit₀</text>
  <text x="708" y="143" text-anchor="middle" font-size="9"  fill="#888">generated</text>
  <line x1="675" y1="155" x2="748" y2="155" stroke="#ddd" stroke-width="1"/>
  <text x="708" y="170" text-anchor="middle" font-size="11" fill="#b71c1c">logit₁</text>
  <text x="708" y="186" text-anchor="middle" font-size="9"  fill="#888">curated</text>

  <!-- Arrows -->
  <defs><marker id="marr" markerWidth="8" markerHeight="8" refX="7" refY="3" orient="auto"><path d="M0,0 L0,6 L8,3 z" fill="#888"/></marker></defs>
  <line x1="110" y1="145" x2="130" y2="145" stroke="#888" stroke-width="2" marker-end="url(#marr)"/>
  <line x1="250" y1="145" x2="270" y2="145" stroke="#888" stroke-width="2" marker-end="url(#marr)"/>
  <line x1="384" y1="145" x2="404" y2="145" stroke="#888" stroke-width="2" marker-end="url(#marr)"/>
  <line x1="518" y1="145" x2="538" y2="145" stroke="#888" stroke-width="2" marker-end="url(#marr)"/>
  <line x1="650" y1="145" x2="666" y2="145" stroke="#888" stroke-width="2" marker-end="url(#marr)"/>

  <!-- CE loss and softmax notes -->
  <text x="380" y="263" text-anchor="middle" font-size="11" fill="#555">
    Training: <tspan font-weight="bold">CrossEntropyLoss</tspan>(logits, labels) with <tspan font-weight="bold">SGD</tspan>
    <tspan dx="20"/>Inference: <tspan font-weight="bold">softmax</tspan>(logits) → probabilities
  </text>
</svg>

In [20]:
class NameMLP(nn.Module):
    # TODO [2 points]: Define the MLP as described above
    def __init__(self, vocab_size=27, embed_dim=16, max_len=MAX_LEN):
        super().__init__()
        self.layers = nn.Sequential(nn.Embedding(vocab_size, embed_dim), nn.Flatten(), nn.Linear(MAX_LEN * embed_dim, 128), nn.ReLU(), nn.Linear(128, 64), nn.ReLU(), nn.Linear(64, 2))

        # TODO: define layers

    def forward(self, input_ids):
        # TODO: implement forward pass
        return self.layers(input_ids)
        pass

### Test your model
The cell below should print shapes matching:
```
Logits shape: torch.Size([4, 2])
Logits dtype: torch.float32
```

In [21]:
model = NameMLP()
print(f"Model:\n{model}\n")

# Test with a small batch
test_names = ['alice', 'xzqwp', 'bob', 'rrrrr']
test_input = torch.stack([tokenize(n) for n in test_names])
print(f"Input shape: {test_input.shape}, dtype: {test_input.dtype}")

logits = model(test_input)
print(f"Logits shape: {logits.shape}")
print(f"Logits dtype: {logits.dtype}")

Model:
NameMLP(
  (layers): Sequential(
    (0): Embedding(27, 16)
    (1): Flatten(start_dim=1, end_dim=-1)
    (2): Linear(in_features=240, out_features=128, bias=True)
    (3): ReLU()
    (4): Linear(in_features=128, out_features=64, bias=True)
    (5): ReLU()
    (6): Linear(in_features=64, out_features=2, bias=True)
  )
)

Input shape: torch.Size([4, 15]), dtype: torch.int64
Logits shape: torch.Size([4, 2])
Logits dtype: torch.float32


## Part 4: Prepare Data and Train [3 points]

**4a** [1 point]: Tokenize all training names and create a `TensorDataset` and `DataLoader`. Also tokenize the validation names (we'll use them for evaluation).

**4b** [2 points]: Write the training loop. Use:
- **Loss**: `nn.CrossEntropyLoss`
- **Optimizer**: `SGD` with learning rate `0.1`
- **Epochs**: 20
- **Batch size**: 128

Print the average training loss every 5 epochs.

**Reminder**: Call `model.train()` before the training loop to set the model to training mode.

In [22]:
# TODO 4a [1 point]: Tokenize all names and create DataLoader
# Hint: use torch.stack to combine individual tokenized tensors

batch_size = 128

# TODO: tokenize all training names into a tensor of shape (16000, MAX_LEN)
train_tokens = torch.stack([tokenize(n) for n in train_names])

# TODO: also tokenize validation names into a tensor of shape (4000, MAX_LEN)
valid_tokens = torch.stack([tokenize(n) for n in val_names])

# TODO: create TensorDataset and DataLoader for training data (with shuffle=True)
train_data = TensorDataset(train_tokens, train_labels)
train_loader = DataLoader(train_data, batch_size, shuffle=True)

In [23]:
# TODO 4b [2 points]: Training loop

n_epochs = 20
learning_rate = 0.1

# TODO: Initialize a fresh model, loss function (CrossEntropyLoss), and optimizer (SGD)
model = NameMLP()
loss_fn = nn.CrossEntropyLoss()
optimizer = SGD(model.parameters(), lr=0.1)
# TODO: Set model to training mode with model.train()
model.train()

# TODO: Training loop
# For each epoch:
#   For each batch from the DataLoader:
#     1. Forward pass: get logits from model
#     2. Compute loss using loss_fn
#     3. Zero gradients with optimizer.zero_grad()
#     4. Backward pass with loss.backward()
#     5. Update weights with optimizer.step()
#   Print average loss every 5 epochs
for epoch in range(n_epochs):
  total_loss = 0
  for batch in train_loader:
    logits = model(batch[0])
    loss = loss_fn(logits, batch[1])
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    total_loss += loss.item()
  if (epoch + 1) % 5 == 0:
    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch + 1}: {avg_loss}")

Epoch 5: 0.5070666823387145
Epoch 10: 0.4535675559043884
Epoch 15: 0.3976499457359314
Epoch 20: 0.3435639303922653


## Part 5: Evaluation and Inference [2 points]

**5a** [1 point]: Compute the accuracy of your trained model on the **validation set**.

**Important**: Before evaluating, call `model.eval()` to set the model to evaluation mode, and wrap your code in `with torch.no_grad():` to disable gradient computation. This is both good practice and necessary for correct behavior with certain layer types (like dropout or batch norm).

Use `torch.argmax` on the logits to get predicted classes, then compare to labels.

**5b** [1 point]: Write a function `name_probability` that takes a name string and returns the probability that it is a **curated** name. Use `torch.softmax` on the logits to convert them to probabilities. Test it on your own name (lowercase!) and a couple of made-up strings.

You should be able to achieve **>75% validation accuracy** with the hyperparameters above. (This task is genuinely hard — the bigram model produces surprisingly name-like outputs!)

In [24]:
# TODO 5a [1 point]: Compute accuracy on the VALIDATION set
# Remember: model.eval() and torch.no_grad()!
model.eval()
with torch.no_grad():
  pred = model(valid_tokens)
  pred = torch.argmax(pred, dim=1)
  val_acc = (pred == val_labels).sum() / len(val_labels)

# TODO: forward pass on val_tokens, get predictions with argmax, compute accuracy
print(f"Validation Accuracy: {val_acc}")

Validation Accuracy: 0.7770000100135803


In [25]:
# TODO 5b [1 point]: Define name_probability and test it

def name_probability(name, model):
    """Return the probability that `name` is a curated name.

    Args:
        name: lowercase string
        model: trained NameMLP
    Returns:
        float: probability of being a curated name (class 1)
    """
    # TODO: set eval mode, use no_grad, tokenize, forward pass,
    # softmax to get probabilities, return prob of class 1
    model.eval()
    with torch.no_grad():
      prob = torch.softmax(model(tokenize(name).unsqueeze(0)), dim=1) # convert to batch size shape
    prob = prob[0][1].item()
    return prob
    pass

# TODO: Test on your own name (lowercase) and at least 2 made-up strings
# Example:
print(f"greg:    {name_probability('greg', model):.4f}")
print(f"vedant:    {name_probability('vedant', model):.4f}")
print(f"xzqwp:   {name_probability('xzqwp', model):.4f}")
print(f"emma:    {name_probability('emma', model):.4f}")

greg:    0.5687
vedant:    0.2995
xzqwp:   0.8988
emma:    0.8475


## Submission
Export this notebook to PDF and submit on Canvas.
```python
!jupyter nbconvert --to pdf --output=yourname_hw0.pdf hw0.ipynb
```